# NLP Project 2

**ESILV A4 DIA6 — 2026**



**Authors:** Leo WINTER & Alvaro SERERO

## Table of Contents   
1. [Setup & Imports](#setup)
2. [Load Data](#load)
3. [Baseline ML model](#model1)
4. [Model with embeddings](#model_keras)
    - 4.1 [Model with trained embeddings](#model2)
    - 4.2 [Model with pre-trained embeddings](#model3)
6. [USE](#model4)
7. [LLM](#model5)
8. [Comparaison](#comparaison)


<a id="setup"></a>
## 1. Setup & Imports

In [2]:
from pathlib import Path
import os, pickle, ast, datetime

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import gensim.downloader
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, GlobalMaxPooling1D,SpatialDropout1D,
                                      Dense, Dropout,Conv1D,BatchNormalization,
                                      LSTM, Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping

from transformers import pipeline


import matplotlib.pyplot as plt
import seaborn as sns



# Setting up the Paths
CURRENT_DIR = Path.cwd()
DATA_PATH = CURRENT_DIR.parent / "data"
MODEL_PATH = CURRENT_DIR.parent / "model"
VISU_PATH = CURRENT_DIR.parent / "visualizations" / "notebook3"
LOG_DIR = CURRENT_DIR.parent / "logs" / "projector"
VISU_PATH.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\leoma\Documents\ESILV\A4_S2\Information Retrieval and NLP\NLP_2\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class TqdmCallback(CallbackAny2Vec):
    def __init__(self, total):
        self.pbar  = tqdm(total=total, desc="Word2Vec epochs")
        self.epoch = 0
    def on_epoch_end(self, model):
        self.epoch += 1
        self.pbar.update(1)
        if self.epoch == self.pbar.total:
            self.pbar.close()


class Word2VecModel:
    """
    Wrapper around gensim Word2Vec providing a clean interface for:
    similarity, distance, analogy, 2D visualisation, sentence vectors,
    and semantic search — matching the GloVe interface below.
    """
    def __init__(self, vector_size=100, window=5, min_count=2, sg=1, epochs=10):
        self.vector_size = vector_size
        self.window, self.min_count = window, min_count
        self.sg, self.epochs = sg, epochs
        self.model = None

    def train(self, sentences):
        cb = TqdmCallback(self.epochs)
        self.model = Word2Vec(
            sentences=sentences, vector_size=self.vector_size,
            window=self.window, min_count=self.min_count,
            workers=4, epochs=self.epochs, sg=self.sg, callbacks=[cb]
        )

    def save(self, name):
        if self.model:
            self.model.save(str(MODEL_PATH / f"{name}.model"))
            print(f"Saved → {name}.model")

    def load(self, name):
        path = MODEL_PATH / f"{name}.model"
        if path.exists():
            self.model = Word2Vec.load(str(path))
            print(f"Loaded ← {name}.model")
        else:
            print(f"Not found: {path}")

    # ── Vocab helpers ─────────────────────────────────────────────────────────
    def __contains__(self, w):   return self.model and w in self.model.wv
    def vec(self, w):            return self.model.wv[w]
    def vocab(self):             return self.model.wv.index_to_key
    def vocab_size(self):        return len(self.model.wv.index_to_key)
    def n_dims(self):            return self.vector_size

    # ── Distances (section 5) ─────────────────────────────────────────────────
    def cosine(self, w1, w2):
        """Cosine similarity between two words (range: −1 to 1, higher = more similar)."""
        if w1 in self and w2 in self:
            v1, v2 = self.vec(w1), self.vec(w2)
            return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
        return None

    def euclidean(self, w1, w2):
        """Euclidean distance between two word vectors (lower = more similar)."""
        if w1 in self and w2 in self:
            return float(np.linalg.norm(self.vec(w1) - self.vec(w2)))
        return None

    # ── Analogy (section 6) ───────────────────────────────────────────────────
    def analogy(self, positive, negative, topn=5):
        """
        Word analogy: find words such that
        positive[0] - negative[0] + positive[1] ≈ result
        Classic example: king - man + woman ≈ queen
        """
        return self.model.wv.most_similar(positive=positive, negative=negative, topn=topn)

    def most_similar(self, word, topn=10):
        return self.model.wv.most_similar(word, topn=topn)

    # ── Visualisation ─────────────────────────────────────────────────────────
    def plot_2d(self, words, title="Word2Vec 2D", fname=None):
        valid  = [w for w in words if w in self]
        if len(valid) < 2: print("Not enough words."); return
        vecs   = np.array([self.vec(w) for w in valid])
        coords = PCA(n_components=2).fit_transform(vecs)
        plt.figure(figsize=(10, 7))
        plt.scatter(coords[:, 0], coords[:, 1], color="#1565C0",
                    edgecolors="white", s=60, zorder=3)
        for i, w in enumerate(valid):
            plt.annotate(w, (coords[i, 0], coords[i, 1]),
                         xytext=(5, 3), textcoords="offset points", fontsize=8)
        plt.title(title, fontweight="bold")
        plt.grid(True, linestyle="--", alpha=0.3); plt.tight_layout()
        if fname: plt.savefig(VISU_PATH / f"{fname}.png", bbox_inches="tight")
        plt.show()

    # ── Sentence vectors (section 8) ──────────────────────────────────────────
    def sentence_vec(self, tokens, idf=None):
        """
        TF-IDF weighted mean of word vectors.
        Rare, informative words get higher weight than frequent generic words.
        Returns None if no known words are found.
        """
        valid = [(self.vec(w), idf.get(w, 1.0) if idf else 1.0)
                 for w in tokens if w in self]
        if not valid: return None
        vecs, ws = zip(*valid)
        return np.average(vecs, axis=0, weights=ws)

    def semantic_search(self, query_tokens, corpus_vecs, corpus_idx, top_n=3, idf=None):
        qv = self.sentence_vec(query_tokens, idf)
        if qv is None: return []
        mat   = np.array(corpus_vecs)
        norms = np.linalg.norm(mat, axis=1) * np.linalg.norm(qv)
        sims  = np.divide(np.dot(mat, qv), norms,
                          out=np.zeros(len(mat)), where=norms != 0)
        top_pos = list(np.argsort(sims)[-top_n:][::-1])
        return [corpus_idx[p] for p in top_pos]

class GloVeModel:
    """
    Wrapper around a pre-trained gensim KeyedVectors object.
    Provides the same interface as Word2VecModel so both can be used
    interchangeably in comparison and search code.
    """
    def __init__(self, name: str):
        cache = MODEL_PATH / f"{name}.vecs"
        if cache.exists():
            print(f"Loading {name} from cache…")
            with open(cache, "rb") as f:
                self.kv = pickle.load(f)
        else:
            print(f"Downloading {name} (first run only, ~130 MB)…")
            self.kv = gensim.downloader.load(name)
            with open(cache, "wb") as f:
                pickle.dump(self.kv, f)
        print(f"Loaded: {len(self.kv.index_to_key):,} words, {self.kv.vector_size} dimensions")

    def __contains__(self, w):   return w in self.kv
    def vec(self, w):            return np.array(self.kv[w])
    def vocab(self):             return self.kv.index_to_key
    def vocab_size(self):        return len(self.kv.index_to_key)
    def n_dims(self):            return self.kv.vector_size
    def most_similar(self, w, topn=10): return self.kv.most_similar(w, topn=topn)

    def cosine(self, w1, w2):
        if w1 in self and w2 in self:
            v1, v2 = self.vec(w1), self.vec(w2)
            return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
        return None

    def euclidean(self, w1, w2):
        if w1 in self and w2 in self:
            return float(np.linalg.norm(self.vec(w1) - self.vec(w2)))
        return None

    def analogy(self, positive, negative, topn=5):
        return self.kv.most_similar(positive=positive, negative=negative, topn=topn)

    def sentence_vec(self, tokens, idf=None):
        valid = [(self.vec(w), idf.get(w, 1.0) if idf else 1.0)
                 for w in tokens if w in self]
        if not valid:
            return np.zeros(self.n_dims())
        vecs, ws = zip(*valid)
        return np.average(vecs, axis=0, weights=ws)

    def semantic_search(self, query_tokens, corpus_vecs, corpus_idx, top_n=3, idf=None):
        qv = self.sentence_vec(query_tokens, idf)
        if np.all(qv == 0): return []
        mat   = np.array(corpus_vecs)
        norms = np.linalg.norm(mat, axis=1) * np.linalg.norm(qv)
        sims  = np.divide(np.dot(mat, qv), norms,
                          out=np.zeros(len(mat)), where=norms != 0)
        top_pos = list(np.argsort(sims)[-top_n:][::-1])
        return [corpus_idx[p] for p in top_pos]

    def plot_2d(self, words, title="GloVe 2D", fname=None):
        valid  = [w for w in words if w in self]
        if len(valid) < 2: return
        vecs   = np.array([self.vec(w) for w in valid])
        coords = PCA(n_components=2).fit_transform(vecs)
        plt.figure(figsize=(10, 7))
        plt.scatter(coords[:, 0], coords[:, 1], color="#0D47A1",
                    edgecolors="white", s=60, zorder=3)
        for i, w in enumerate(valid):
            plt.annotate(w, (coords[i, 0], coords[i, 1]),
                         xytext=(5, 3), textcoords="offset points", fontsize=8)
        plt.title(title, fontweight="bold")
        plt.grid(True, linestyle="--", alpha=0.3); plt.tight_layout()
        if fname: plt.savefig(VISU_PATH / f"{fname}.png", bbox_inches="tight")
        plt.show()


model_glove = GloVeModel("glove-wiki-gigaword-100")


Loading glove-wiki-gigaword-100 from cache…
Loaded: 400,000 words, 100 dimensions


<a id="load"></a>
## 2. Load Data

In [3]:
# Load data saved in the second notebook
def load_data():
    path_parquet = DATA_PATH / "reviews_modeled.parquet"
    path_csv = DATA_PATH / "reviews_modeled.csv"

    if path_parquet.exists():
        try:
            df = pd.read_parquet(path_parquet)
            return df
        except Exception as e:
            print(e)

    if path_csv.exists():
        try:
            df = pd.read_csv(path_csv)
            return df
        except Exception as e:
            print(e)
    return None


dataset = load_data()
if dataset is None: 
    dataset = pd.DataFrame() #To not have None warnings
display(dataset.head(2))
data_model = dataset.copy()

,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en,year_month,tokens_en,tokens,cluster_en,cluster_fr
0,5,brahim--k-131532,"Meilleurs assurances, prix, solutions, écoute,...",Direct Assurance,auto,train,2021-09-06,2021-09-01,"Best insurance, price, solutions, listening, s...",meilleurs assurances prix solutions écoute rap...,best insurance price solutions listening speed...,2021-09,"['good', 'price', 'solution', 'listen', 'speed...","['meilleur', 'prix', 'solution', 'écoute', 'ra...",1,1
1,4,bernard-g-112497,"je suis globalement satisfait , sauf que vous ...",Direct Assurance,auto,train,2021-05-03,2021-05-01,"I am generally satisfied, except that you have...",je suis globalement satisfait sauf que vous av...,i am generally satisfied except that you have ...,2021-05,"['generally', 'satisfied', 'except', 'problem'...","['globalement', 'satisfait', 'sauf', 'problème...",0,2


In [4]:
def label_sentiment(rating):
    if rating <= 2: return 0 
    if rating == 3: return 1 
    return 2                 

#data_model['emotion'] = data_model['note'].apply(label_sentiment)
X_en = data_model['tokens_en']
X_fr = data_model['tokens']
y = data_model['note']
y = y - 1 # To have labels from 0 to 4 and not from 1 to 5

X_train_en, X_test_en, y_train_en, y_test_en = train_test_split(X_en, y, test_size=0.2, random_state=42, stratify=y)
X_train_fr, X_test_fr, y_train_fr, y_test_fr = train_test_split(X_fr, y, test_size=0.2, random_state=42, stratify=y)

X_train_list_en = [ast.literal_eval(x) for x in X_train_en.tolist()]
X_test_list_en = [ast.literal_eval(x) for x in X_test_en.tolist()]

X_train_list_fr = [ast.literal_eval(x) for x in X_train_fr.tolist()]
X_test_list_fr = [ast.literal_eval(x) for x in  X_test_fr.tolist()]


In [6]:
X_train_list_en

[['file',
  'quickly',
  'complete',
  'website',
  'yet',
  'computer',
  'pro',
  'rate',
  'attractive',
  'highly',
  'recommend'],
 ['satisfied',
  'lose',
  'follow',
  'divorce',
  'despite',
  'intervention',
  'explain',
  'delay',
  'payment',
  'way',
  'fault',
  'wife',
  'systematically',
  'throw',
  'mail',
  'trash',
  'try',
  'learn',
  'terminate',
  'contract',
  'generate',
  'great',
  'precariousness',
  'divorce',
  'put',
  'yet',
  'year',
  'contract',
  'sinister',
  'declare',
  'delay',
  'payment',
  'take',
  'small',
  'problem',
  'could',
  'resolve',
  'see',
  'contract',
  'number'],
 ['disappointed',
  'mutual',
  'take',
  'monthly',
  'payment',
  'import',
  'sometimes',
  'several',
  'time',
  'hurt',
  'wallet',
  'subscription',
  'make',
  'error',
  'date',
  'blow',
  'deduct',
  'month',
  'february',
  'subscribe',
  'take',
  'november',
  'january',
  'course',
  'course',
  'want',
  'reimburse',
  'continue',
  'take',
  'leave'],

In [87]:
model_w2v_en = Word2VecModel(vector_size=100, window=5, min_count=2, sg=1, epochs=10)
model_w2v_fr = Word2VecModel(vector_size=100, window=5, min_count=2, sg=1, epochs=10)
model_glove = GloVeModel("glove-wiki-gigaword-100")

if (MODEL_PATH / "word2vec_en.model").exists():
    model_w2v_en.load("word2vec_en")
else:
    print("W2v English model not found")
if (MODEL_PATH / "word2vec_fr.model").exists():
    model_w2v_fr.load("word2vec_fr")
else:
    print("W2v French model not found")

Loading glove-wiki-gigaword-100 from cache…
Loaded: 400,000 words, 100d
Loaded ← word2vec_en.model
Loaded ← word2vec_fr.model


<a id="model1"></a>
## 3. Baseline ML model

In [27]:
def model1(X_train, X_test, y_train, y_test):
    tfidf = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=20_000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    model_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
    model_lr.fit(X_train_tfidf, y_train)

    y_pred = model_lr.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    print(classification_report(y_test, y_pred))
    
    return tfidf,model_lr,accuracy

In [28]:
print("English model note:")
tfidf_en, model_lr_en,accuracy1_en = model1(X_train_en, X_test_en, y_train_en, y_test_en)
print("\nFrench model note:")
tfidf_fr, model_lr_fr,accuracy1_fr = model1(X_train_fr, X_test_fr, y_train_fr, y_test_fr)

English model note:
              precision    recall  f1-score   support

           0       0.67      0.66      0.66      1453
           1       0.35      0.38      0.36       743
           2       0.28      0.29      0.28       677
           3       0.42      0.38      0.40       977
           4       0.54      0.55      0.54       969

    accuracy                           0.49      4819
   macro avg       0.45      0.45      0.45      4819
weighted avg       0.49      0.49      0.49      4819


French model note:
              precision    recall  f1-score   support

           0       0.67      0.67      0.67      1453
           1       0.36      0.40      0.38       743
           2       0.31      0.29      0.30       677
           3       0.46      0.42      0.44       977
           4       0.56      0.58      0.57       969

    accuracy                           0.50      4819
   macro avg       0.47      0.47      0.47      4819
weighted avg       0.51      0.50    

In [29]:
def plot_top_coefficients(vectorizer, model, n_words=10):
    feature_names = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]
    
    top_pos = np.argsort(coefs)[-n_words:]
    top_neg = np.argsort(coefs)[:n_words]
    
    print("Important negative worlds:", [feature_names[i] for i in top_neg])
    print("Important positive worlds:", [feature_names[i] for i in top_pos])

print("English note:")
print(f"accuracy = {accuracy1_en:.2f}")
plot_top_coefficients(tfidf_en, model_lr_en)
print("\nFrench note:")
print(f"accuracy = {accuracy1_fr:.2f}")
plot_top_coefficients(tfidf_fr, model_lr_fr)


English note:
accuracy = 0.49
Important negative worlds: ['satisfied', 'thank', 'good', 'price', 'fast', 'moment', 'quick', 'easy', 'hope', 'satisfied service']
Important positive worlds: ['expensive', 'impossible', 'reimburse', 'pay', 'terminate', 'month', 'incompetent', 'increase', 'avoid', 'flee']

French note:
accuracy = 0.50
Important negative worlds: ['satisfait', 'satisfaite', 'rapide', 'merci', 'prix', 'très', 'espère', 'écoute', 'bon', 'efficace']
Important positive worlds: ['argent', 'incompétent', 'nul', 'aucun', 'déconseille', 'aucune', 'éviter', 'fuyez', 'mois', 'fuir']


<a id="model_keras"></a>
## 4. Model with embeddings keras

In [ ]:
max_words = 10000  
max_length = 100
vector_size=100

tokenizer_en = Tokenizer(num_words=max_words, oov_token="<OOV>",filters='',lower=False)
tokenizer_en.fit_on_texts(X_train_list_en)

X_train_seq_en = tokenizer_en.texts_to_sequences(X_train_list_en)
X_test_seq_en = tokenizer_en.texts_to_sequences(X_test_list_en)
X_train_pad_en = pad_sequences(X_train_seq_en, maxlen=max_length, padding='post')
X_test_pad_en = pad_sequences(X_test_seq_en, maxlen=max_length, padding='post')


tokenizer_fr = Tokenizer(num_words=max_words, oov_token="<OOV>", filters='',lower=False)
tokenizer_fr.fit_on_texts(X_train_list_fr)

X_train_seq_fr = tokenizer_fr.texts_to_sequences(X_train_list_fr)
X_test_seq_fr = tokenizer_fr.texts_to_sequences(X_test_list_fr)
X_train_pad_fr = pad_sequences(X_train_seq_fr, maxlen=max_length, padding='post')
X_test_pad_fr = pad_sequences(X_test_seq_fr, maxlen=max_length, padding='post')

def get_embedding_matrix(tokenizer, word2vec_model, embedding_dim):
    vocab_size = len(tokenizer.word_index) + 1
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    
    for word, i in tokenizer.word_index.items():
        if word in word2vec_model:
            embedding_matrix[i] = word2vec_model.vec(word)
            
    return embedding_matrix

<a id="model2"></a>
### 4.1 Model with trained embeddings 

In [157]:
def model2_en(embedding_matrix,embedding_dim = 100):
    vocab_size, embedding_dim = embedding_matrix.shape
   
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix],
                  trainable=False),
                  
        SpatialDropout1D(0.5),
        Conv1D(128, 5, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        BatchNormalization(), 
        
        Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.0001)),

        Dropout(0.5),
        Dense(5, activation='softmax')
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
    model.compile(loss='sparse_categorical_crossentropy', 
                      optimizer=optimizer, 
                      metrics=['accuracy'])
    return model

def model2_fr(embedding_matrix,embedding_dim = 100):
    vocab_size, embedding_dim = embedding_matrix.shape
   
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix],
                  trainable=False),
                  
        SpatialDropout1D(0.2),
        Conv1D(128, 5, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        BatchNormalization(), 
        
        Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.0001)),

        Dropout(0.5),
        Dense(5, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', 
                      optimizer='adam', 
                      metrics=['accuracy'])
    return model

In [158]:
embedding_matrix_en = get_embedding_matrix(tokenizer_en, model_w2v_en, vector_size)
embedding_matrix_fr = get_embedding_matrix(tokenizer_fr, model_w2v_fr, vector_size)

model_embeddings_en = model2_en(embedding_matrix_en)
model_embeddings_en.build(input_shape=(None, max_length))
model_embeddings_en.summary()

model_embeddings_fr = model2_fr(embedding_matrix_fr)

Model: "sequential_41"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_41 (Embedding)        │ (None, 100, 100)       │     1,135,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_12            │ (None, 100, 100)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_45 (Conv1D)              │ (None, 100, 128)       │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_43         │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_82 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_41 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_83 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,208,621 (4.61 MB)

 Trainable params: 72,965 (285.02 KB)

 Non-trainable params: 1,135,656 (4.33 MB)

In [161]:
log_dir = str(LOG_DIR / "fit" / f"{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10,          
    restore_best_weights=True 
)

print("English model :")
history = model_embeddings_en.fit(
    X_train_pad_en, y_train_en,
    epochs=100,
    validation_data=(X_test_pad_en, y_test_en),
    callbacks=[tensorboard_callback,early_stop],
    batch_size=32
)


print("\nFrench model :")
history = model_embeddings_fr.fit(
    X_train_pad_fr, y_train_fr,
    epochs=100,
    validation_data=(X_test_pad_fr, y_test_fr),
    callbacks=[tensorboard_callback,early_stop],
    batch_size=32
)

English model :
Epoch 1/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.3136 - loss: 1.8516 - val_accuracy: 0.4397 - val_loss: 1.3139
Epoch 2/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.3797 - loss: 1.5171 - val_accuracy: 0.4565 - val_loss: 1.2618
Epoch 3/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.4009 - loss: 1.4314 - val_accuracy: 0.4630 - val_loss: 1.2316
Epoch 4/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.4187 - loss: 1.3695 - val_accuracy: 0.4804 - val_loss: 1.2066
Epoch 5/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.4207 - loss: 1.3274 - val_accuracy: 0.4864 - val_loss: 1.1949
Epoch 6/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.4337 - loss: 1.2947 - val_accuracy: 0.4866 - val_loss: 1.1895
Epoch 7/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.4463 - loss: 1.2705 - val_accuracy: 0.4868 - val_loss: 1.1796
Epoch 8/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.45

In [162]:
loss_en, accuracy2_en = model_embeddings_en.evaluate(X_test_pad_en, y_test_en,verbose=0)
loss_fr, accuracy2_fr = model_embeddings_fr.evaluate(X_test_pad_fr, y_test_fr,verbose=0)

print("English model :")
print(f"Loss     : {loss_en:.4f}")
print(f"Accuracy : {accuracy2_en:.2}")
print("\nFrench model :")
print(f"Loss     : {loss_fr:.4f}")
print(f"Accuracy : {accuracy2_fr:.2}")

English model :
Loss     : 1.0820
Accuracy : 0.52

French model :
Loss     : 1.0586
Accuracy : 0.52


<a id="model3"></a>
### 4.2 Model with pre-trained embeddings

In [136]:
def model3(embedding_matrix,embedding_dim = 100):
    vocab_size, embedding_dim = embedding_matrix.shape
   
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix],
                  trainable=True),
        SpatialDropout1D(0.5),
        Conv1D(64, 5, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        BatchNormalization(), 
        
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.0001)),
        Dropout(0.5),
        Dense(5, activation='softmax')
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
    model.compile(loss='sparse_categorical_crossentropy', 
                      optimizer=optimizer, 
                      metrics=['accuracy'])
    return model

In [137]:
embedding_matrix = get_embedding_matrix(tokenizer_en, model_glove, vector_size)

model_embeddings = model3(embedding_matrix)
model_embeddings.build(input_shape=(None, max_length))
model_embeddings.summary()

Model: "sequential_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_32 (Embedding)        │ (None, 100, 100)       │     1,135,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_3             │ (None, 100, 100)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_36 (Conv1D)              │ (None, 100, 64)        │        32,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_34         │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_65 (Dense)                │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,169,965 (4.46 MB)

 Trainable params: 1,169,837 (4.46 MB)

 Non-trainable params: 128 (512.00 B)

In [138]:
log_dir = str(LOG_DIR / "fit" / f"{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5,          
    restore_best_weights=True 
)

history = model_embeddings.fit(
    X_train_pad_en, y_train_en,
    epochs=100,
    validation_data=(X_test_pad_en, y_test_en),
    callbacks=[tensorboard_callback,early_stop],
    batch_size=32
)

Epoch 1/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.2360 - loss: 2.0033 - val_accuracy: 0.2687 - val_loss: 1.5662
Epoch 2/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.3247 - loss: 1.6433 - val_accuracy: 0.3314 - val_loss: 1.4717
Epoch 3/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.3502 - loss: 1.5317 - val_accuracy: 0.3725 - val_loss: 1.4246
Epoch 4/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.3770 - loss: 1.4638 - val_accuracy: 0.3970 - val_loss: 1.3949
Epoch 5/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.3924 - loss: 1.4166 - val_accuracy: 0.4204 - val_loss: 1.3592
Epoch 6/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.4060 - loss: 1.3748 - val_accuracy: 0.4314 - val_loss: 1.3357
Epoch 7/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.4131 - loss: 1.3616 - val_accuracy: 0.4370 - val_loss: 1.3218
Epoch 8/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.4244 - loss: 1.3365 -

In [139]:
loss, accuracy2_bis = model_embeddings.evaluate(X_test_pad_en, y_test_en,verbose=0)

print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {accuracy2_bis:.2}")

Loss     : 1.0899
Accuracy : 0.51


<a id="model4"></a>
## 5. BERT model

<a id="model_bert"></a>
### 5.2 BERT

<a id="model5"></a>
## 7. LLM

<a id="comparaison"></a>
## 8. Comparaison